In [26]:
import pandas as pd
import numpy as np
import re
import unicodedata
from difflib import SequenceMatcher
from collections import defaultdict
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None) # Desactivamos el límite de columnas para que las muestre todas
pd.set_option('display.max_colwidth', None)

In [2]:
# 1. Cargamos el dataset original y mostramos las primeras 5 filas
df = pd.read_csv('games.csv')
df.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 125855 entries, 2539430 to 4247310
Data columns (total 39 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125854 non-null  str    
 1   Name                        125855 non-null  str    
 2   Release date                125855 non-null  str    
 3   Estimated owners            125855 non-null  int64  
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  float64
 6   Price                       125855 non-null  int64  
 7   DiscountDLC count           125855 non-null  int64  
 8   About the game              117393 non-null  str    
 9   Supported languages         125855 non-null  str    
 10  Full audio languages        125855 non-null  str    
 11  Reviews                     12181 non-null   str    
 12  Header image                125774 non-null  str    
 13  Website                

In [4]:
df.describe()

,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,Metacritic score,User score,Positive,Negative,Score rank,Achievements,Recommendations,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Movies
count,1.258550e+05,125855.000000,125855.000000,125855.000000,125855.000000,125855.000000,125855.000000,1.258550e+05,1.258550e+05,40.000000,125855.000000,1.258550e+05,1.258550e+05,125855.000000,1.258550e+05,125855.000000,0.0
mean,5.318720e+01,0.165365,4.811174,17.896826,0.535434,2.500083,0.023916,1.018182e+03,1.648786e+02,99.175000,17.922220,9.380399e+02,2.027568e+02,13.507711,1.691803e+02,14.419125,NaN
std,3.681083e+03,1.642666,12.489329,28.628518,14.328417,13.492922,1.376811,2.772782e+04,5.305001e+03,0.675107,139.696425,2.159585e+04,1.107222e+04,266.925759,1.105725e+04,290.747134,NaN
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,98.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
25%,0.000000e+00,0.000000,0.590000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,99.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
50%,0.000000e+00,0.000000,2.390000,0.000000,0.000000,0.000000,0.000000,4.000000e+00,1.000000e+00,99.000000,2.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
75%,0.000000e+00,0.000000,5.545000,37.000000,0.000000,0.000000,0.000000,3.500000e+01,1.000000e+01,100.000000,19.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
max,1.013936e+06,21.000000,999.980000,100.000000,3703.000000,97.000000,100.000000,7.642084e+06,1.173003e+06,100.000000,9821.000000,4.830455e+06,3.429544e+06,20088.000000,3.429544e+06,20088.000000,NaN


In [5]:
df.isnull().sum()

AppID                              1
Name                               0
Release date                       0
Estimated owners                   0
Peak CCU                           0
Required age                       0
Price                              0
DiscountDLC count                  0
About the game                  8462
Supported languages                0
Full audio languages               0
Reviews                       113674
Header image                      81
Website                        75335
Support url                    70560
Support email                  22630
Windows                            0
Mac                                0
Linux                              0
Metacritic score                   0
Metacritic url                121597
User score                         0
Positive                           0
Negative                           0
Score rank                    125815
Achievements                       0
Recommendations                    0
N

Sección: Preprocesamiento y Control de Calidad del Dataset
1. Diagnóstico de Anomalías en la Ingesta de Datos


Durante la fase inicial de carga del archivo games.csv mediante la librería pandas, se detectó un desalineamiento crítico en la matriz de datos. El síntoma principal fue el desplazamiento de los valores hacia la derecha a partir de la columna Price, lo que provocaba inconsistencias severas en el dominio de las variables (por ejemplo, fechas de lanzamiento indexadas como nombres de videojuegos, o registros numéricos incoherentes en la restricción de edad, como Required age = 64).

2. Análisis Técnico de la Causa Raíz

Para identificar el origen del desalineamiento, se realizó una inspección quirúrgica del archivo en texto plano (I/O puro de Python), aislando la estructura de la cabecera (header) respecto a las filas de registros físicos.

El análisis reveló un error de diseño estructural en el dataset de origen: el creador omitió una coma delimitadora en la cadena de la cabecera entre las variables Discount y DLC count, resultando en el string continuo DiscountDLC count.

Impacto técnico: La cabecera corrupta declaraba un total de 39 columnas, mientras que las filas de datos contenían en realidad 40 valores delimitados por comas. Al realizar el parsing estándar, el motor de Pandas sufría un desajuste posicional (data shifting), arrastrando cada celda una posición a la derecha a partir de la séptima columna.

3. Estrategia de Mitigación e Implementación

Para subsanar esta anomalía sin alterar la integridad del archivo original ni perder registros mediante técnicas drásticas de descarte, se diseñó un pipeline de ingesta personalizado:

Inyección de Cabecera Corregida: Se ignoró la línea de cabecera nativa del CSV mediante el parámetro header = 0 y se definió un array explícito de 40 columnas corregidas, restituyendo la separación formal entre Discount y DLC count.

Control de Comas Anidadas: Se forzó el uso del parámetro quotechar = '"' combinado con skipinitialspace = True para asegurar que las fechas con formato anglosajón (ej. "Aug 1, 2023") fueran interpretadas como un único bloque indivisible de texto y no como delimitadores de columna.

Filtrado Selectivo (Feature Selection Prerrequisito): Para optimizar la memoria volátil del entorno y evitar el ruido de strings masivos, se restringió la carga únicamente a las 12 variables críticas identificadas para el modelo de preventa.

In [6]:
# Mostramos las 2 primeras líneas reales del archivo en texto puro para analizar los datos y arreglarlos a mano
with open('games.csv', 'r', encoding = 'utf-8', errors = 'ignore') as f:
    cabecera = f.readline()
    primera_fila = f.readline()
print("--- CABECERA ORIGINAL DEL ARCHIVO ---")
print(cabecera)
print("\n--- PRIMERA FILA REAL DE DATOS ---")
print(primera_fila)

--- CABECERA ORIGINAL DEL ARCHIVO ---
AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies


--- PRIMERA FILA REAL DE DATOS ---
2539430,"Black Dragon Mage Playtest","Aug 1, 2023","0 - 0",0,0,0.0,0,0,"","[]","[]","","https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/header.jpg?t=1699268702","","","",True,False,False,0,"",0,0,0,"",0,0,"",0,0,0,0,"","","","","","https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/ss_7d9504b958d0b143d053d31cb74b375daba338d6.1920x1080.jpg?t=1699268702,https://shared.akamai.steamstat

In [7]:
# 2. Limpieza básica y asignación de índice. Definimos la lista de columnas corregida (reparando el error del creador del dataset)
columnas_corregidas = [
    'AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 
    'Required age', 'Price', 'Discount', 'DLC count', 'About the game', 
    'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 
    'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 
    'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 
    'Score rank', 'Achievements', 'Recommendations', 'Notes', 
    'Average playtime forever', 'Average playtime two weeks', 
    'Median playtime forever', 'Median playtime two weeks', 
    'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 
    'Screenshots', 'Movies'
]

# Cargamos solo las columnas seguras que decidimos usar para la preventa
columnas_a_usar = [
    'Name', 'Release date', 'Estimated owners', 'Required age', 'Price', 
    'Windows', 'Mac', 'Linux', 'Developers', 'Publishers', 'Genres', 'Supported languages'
]

# Leemos el CSV saltándonos la cabecera rota (header = 0) y asignando nuestros nombres corregidos (names = columnas_corregidas)
df_bueno = pd.read_csv('games.csv', header = 0, names = columnas_corregidas, usecols = columnas_a_usar,
    quotechar = '"', skipinitialspace = True, on_bad_lines = 'skip'
)

# Limpiamos los datos y asignamos Name al índice
df_bueno.dropna(subset = ['Name'], inplace = True)
df_bueno.drop_duplicates(subset = ['Name'], inplace = True)
df_bueno.set_index('Name', inplace = True)
df_bueno.head()

,Release date,Estimated owners,Required age,Price,Supported languages,Windows,Mac,Linux,Developers,Publishers,Genres
Name,,,,,,,,,,,
Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0.00,[],True,False,False,NaN,NaN,NaN
Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,5.24,['English'],True,False,False,minori,MangaGamer,Adventure
Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,4.99,"['English', 'French', 'German', 'Russian']",True,True,False,Somer Games,8floor,Casual
버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,0,8.99,['Korean'],True,False,False,유진게임즈,유진게임즈,"Casual,Indie,Simulation"
Maze Quest VR,"Apr 24, 2025",0 - 20000,0,4.99,['English'],True,False,False,Reality Expanded LLC,Reality Expanded LLC,"Action,Early Access"


In [8]:
# 3. Creación del Target Multiclase. Función para limpiar el rango de texto y obtener el promedio numérico de owners
def transformar_owners_a_numerico(rango):
    if pd.isna(rango) or not isinstance(rango, str):
        return 0
    partes = rango.replace(',', '').split(' - ') # Limpiamos comas y separamos por el guion
    if len(partes) == 2:
        try:
            min_owners = float(partes[0])
            max_owners = float(partes[1])
            return (min_owners + max_owners) / 2
        except:
            return 0
    return 0

# Aplicamos la transformación para tener una métrica continua de ventas
df_bueno['owners_promedio'] = df_bueno['Estimated owners'].apply(transformar_owners_a_numerico)
condiciones = [ # Convertimos el valor numérico en las 3 clases de éxito
    (df_bueno['owners_promedio'] < 20000),
    (df_bueno['owners_promedio'] >= 20000) & (df_bueno['owners_promedio'] <= 200000),
    (df_bueno['owners_promedio'] > 200000)
]
df_bueno['exito_target'] = np.select(condiciones, [0, 1, 2], default = 0) # Asignamos las etiquetas (0, 1, 2)
print("Proporción de juegos por clase de éxito:")
print(df_bueno['exito_target'].value_counts(normalize = True) * 100)

Proporción de juegos por clase de éxito:
exito_target
0    79.674112
1    16.065755
2     4.260133
Name: proportion, dtype: float64


Sección: Definición y Formalización de la Variable Objetivo (Target)
1. Justificación Académica del Cambio de Target.

Inicialmente, el éxito de un videojuego se evaluaba mediante métricas cualitativas secundarias (como el ratio de reseñas positivas). Sin embargo, para dotar al proyecto de un enfoque comercial y de negocio riguroso, se redefinió la variable objetivo utilizando Estimated owners (Volumen estimado de propietarios) como el proxy definitivo de éxito de mercado.Predecir directamente las copias vendidas mediante una regresión lineal tradicional introduce un alto grado de variabilidad debido al comportamiento de los outliers (juegos virales o megaproducciones). Por lo tanto, se optó por un enfoque de Clasificación Multiclase, estratificando el mercado en tres segmentos competitivos bien diferenciados.

2. Criterios de Estratificación y Clases de Éxito.

Para evitar umbrales arbitrarios, se adoptaron los estándares de distribución de la industria y las recomendaciones del equipo docente, dividiendo el volumen continuo de propietarios en tres niveles indexados:
   Clase 0: Fracaso Comercial ($<20,000$ propietarios). Define el umbral crítico de la "larga cola" de Steam, donde se encuentra la gran mayoría de los proyectos independientes que no logran recuperar la inversión de desarrollo básica.
   Clase 1: Éxito Medio ($20,000 - 200,000$ propietarios). Representa la clase media de la plataforma; proyectos con una tracción comercial saludable, habitualmente asociados a estudios indies medianos o juegos de nicho de presupuesto moderado (AA)
   Clase 2: Gran Éxito ($>200,000$ propietarios). Identifica a los grandes dinamizadores del mercado: superproducciones AAA y fenómenos virales que logran un impacto masivo en ventas y sostenibilidad a largo plazo.

3. Tratamiento Matemático de Rangos.

Dado que la variable original se almacena como un rango categórico de texto (ej. "20000 - 50000"), el pipeline aplica una función de tipado que extrae los límites inferior y superior, calculando su punto medio aritmético como valor numérico continuo antes de realizar la segmentación por condiciones lógicas.

In [9]:
# 4. Feaute Engineering. Creación de variables. 
# Variables de Conteo Estructurado. Contamos cuántos idiomas y géneros tiene el juego (proxies de presupuesto y alcance)
# Usamos .fillna('') para convertir los valores nulos (NaN) en textos vacíos para evitar errores
# Aplicamos una función lambda que separa el texto por comas mediante .split(',') y averiguar el número de idiomas y géneros
df_bueno['num_idiomas'] = df_bueno['Supported languages'].fillna('').apply(lambda x: len(str(x).split(',')))
df_bueno['num_generos'] = df_bueno['Genres'].fillna('').apply(lambda x: len(str(x).split(',')))

# Convertimos las fechas para extraer mes y año. errors = 'coerce' pasa las fechas rotas a nulo sin romper el código
df_bueno['Release date'] = pd.to_datetime(df_bueno['Release date'], errors = 'coerce')
df_bueno['año_lanzamiento'] = df_bueno['Release date'].dt.year
df_bueno['mes_lanzamiento'] = df_bueno['Release date'].dt.month.fillna(1).astype(int)

# Variable derivada: Lanzamiento en temporada navideña/fiestas (Noviembre y Diciembre)
df_bueno['lanzamiento_navidad'] = df_bueno['mes_lanzamiento'].isin([11, 12]).astype(int)

# Mapeo de Experiencia y Tamaño del Estudio (Indie/AA/AAA). Contamos cuántos juegos ha publicado cada Publisher en todo el dataset histórico
dict_experiencia_pub = df_bueno['Publishers'].value_counts().to_dict()
df_bueno['juegos_publicados_historico'] = df_bueno['Publishers'].map(dict_experiencia_pub).fillna(1).astype(int)

# Clasificación del Estudio basada en su volumen de lanzamientos: 0 = Indie (1 a 9 juegos) | 1 = AA (10 a 50 juegos) | 2 = AAA (>50 juegos)
condiciones_estudio = [
    (df_bueno['juegos_publicados_historico'] > 50),
    (df_bueno['juegos_publicados_historico'] >= 10) & (df_bueno['juegos_publicados_historico'] <= 50),
    (df_bueno['juegos_publicados_historico'] < 10)
]
df_bueno['tipo_estudio'] = np.select(condiciones_estudio, [2, 1, 0], default = 0)

# Aseguramos el tipado numérico del Precio
df_bueno['Price'] = pd.to_numeric(df_bueno['Price'], errors = 'coerce').fillna(0.0)
df_bueno['Required age'] = pd.to_numeric(df_bueno['Required age'], errors = 'coerce').fillna(0).astype(int)
df_bueno.head()

,Release date,Estimated owners,Required age,Price,Supported languages,Windows,Mac,Linux,Developers,Publishers,Genres,owners_promedio,exito_target,num_idiomas,num_generos,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,juegos_publicados_historico,tipo_estudio
Name,,,,,,,,,,,,,,,,,,,,
Black Dragon Mage Playtest,2023-08-01,0 - 0,0,0.00,[],True,False,False,NaN,NaN,NaN,0.0,0,1,1,2023,8,0,1,0
Supipara - Chapter 1 Spring Has Come!,2016-07-29,0 - 20000,0,5.24,['English'],True,False,False,minori,MangaGamer,Adventure,10000.0,0,1,1,2016,7,0,77,2
Mystery Solitaire The Black Raven,2019-05-06,0 - 20000,0,4.99,"['English', 'French', 'German', 'Russian']",True,True,False,Somer Games,8floor,Casual,10000.0,0,4,1,2019,5,0,267,2
버튜버 파라노이아 - Vtuber Paranoia,2024-10-31,0 - 20000,0,8.99,['Korean'],True,False,False,유진게임즈,유진게임즈,"Casual,Indie,Simulation",10000.0,0,1,3,2024,10,0,13,1
Maze Quest VR,2025-04-24,0 - 20000,0,4.99,['English'],True,False,False,Reality Expanded LLC,Reality Expanded LLC,"Action,Early Access",10000.0,0,1,2,2025,4,0,1,0


Sanitización de Datos y Manejo de Valores Ausentes:
Para asegurar la compatibilidad con los algoritmos de Machine Learning, los valores nulos (NaN / NaT) fueron imputados bajo criterios lógicos de negocio: las anomalías en precios y restricciones de edad se fijaron en 0.0 (gratuito) y 0 (sin restricción) respectivamente; las inconsistencias en fechas se asignaron por defecto al mes 1 (enero); y los registros sin historial de distribuidor (Publisher) se inicializaron en 1, clasificándose automáticamente como desarrollos independientes (Indie).

Tratamiento y Normalización de Entidades (Developers):
En los datasets de plataformas de distribución digital (como Steam), el campo del desarrollador presenta una elevada cardinalidad y un alto nivel de ruido debido a discrepancias en el registro de nombres, fusiones de estudios, co-desarrollos y marcas regionales. Para garantizar que las variables de entrada aporten una señal limpia al modelo de aprendizaje automático, se diseñó e implementó un pipeline de procesamiento en tres fases:

1. Limpieza Textual y Eliminación de "Stop Words" de la Industria
En primera instancia, se aplicó una función de limpieza mediante expresiones regulares para estandarizar los nombres a formato Title Case y eliminar caracteres especiales.Asimismo, se identificó que términos genéricos de la industria como «Studios», «Games», «Interactive» o «Software» generaban falsas variaciones de una misma entidad. Tras extirpar estas palabras vacías (stop words específicas del dominio), se logró reducir significativamente el ruido contextual de los datos.

2. Detección de Duplicados mediante Similitud Difusa (Fuzzy Matching)
Para identificar variaciones ortográficas y pequeñas erratas de tipografía, se empleó un algoritmo de comparación difusa basado en la métrica de SequenceMatcher (difflib), utilizando un índice inverso por palabras clave para optimizar la complejidad computacional $O(N^2)$.
- Ajuste del Umbral: Al trabajar sobre texto previamente filtrado, se incrementó el umbral de similitud al 82% ($0.82$), evitando la inclusión de falsos positivos (como diferenciar estudios independientes con nombres similares).
- Escudo Numérico: Se integró una regla de bloqueo mediante expresiones regulares para evitar la unificación de entregas secuenciales o versiones numeradas.

3. Consolidación Semántica de Marcas y Colaboraciones
Dado que la comparación puramente textual no detecta relaciones corporativas ni co-desarrollos, se realizó una auditoría manual de los pares candidatos para aplicar una consolidación de marca:
- Consolidación de Marcas Principales: Múltiples variaciones regionales o de distribución (p. ej., Capcom Vancouver, Capcom USA, Capcom,Gogcom) se unificaron bajo la marca principal (Capcom).
- Atribución de Colaboraciones y Crossovers: En casos de grandes editoras o colectivos prolíficos (como Koei Tecmo, MicroProse, Cyanide o Sokpop Collective), las entradas compuestas por co-desarrollos se atribuyeron a la entidad principal para evitar la fragmentación de la autoría y la aparición de categorías testimoniales (long tail).

4. Codificación Numérica e Ingeniería de Características (Feature Engineering)
Los modelos de aprendizaje automático no pueden interpretar directamente cadenas de texto. Para transformar la variable categórica developer_clean en características numéricas ejecutables sin sufrir los problemas de alta dimensionalidad del One-Hot Encoding, se crearon dos nuevas variables predictoras:
- dev_volumen_juegos ($N$): Variable discreta que contabiliza el número histórico de títulos publicados por el desarrollador en la plataforma, representando la experiencia y presencia en el mercado.
- dev_exito_promedio: Variable continua en el rango $[0, 2]$ que mide la reputación y rendimiento histórico del desarrollador.

5. Suavizado Bayesiano (Bayesian Target Encoding)
Para evitar el sobreajuste (overfitting) en desarrolladores con muy pocos juegos (p. ej., un desarrollador con un único juego de éxito no debe recibir una puntuación perfecta de 2.0 por azar), se implementó un Target Encoding con Suavizado Bayesiano:
$$\text{Éxito Ajustado} = \frac{(N \times \text{Éxito Dev}) + (m \times \text{Éxito Global})}{N + m}$$
Donde $N$ es el volumen de juegos del desarrollador, $\text{Éxito Global}$ es la media del target en todo el dataset, y $m$ es el parámetro de suavizado (fijado en $m = 5$).
- Gestión de Desarrolladores Nuevos en Producción ($N = 0$): para garantizar la capacidad de generalización del modelo ante desarrolladores novatos o sin historial previo en la fase de inferencia, la fórmula asigna automáticamente la media global de la industria (dev_exito_promedio = $\text{Media Global}$) y establece dev_volumen_juegos = $0$. De este modo, se evita penalizar arbitrariamente con un cero a un creador emergente, tratándolo como una incertidumbre neutral ponderada por el mercado.

In [10]:
# 5. Limpieza completa y normalización de los datos para lectura del modelo. Usamos 'Capcom' para demostrar que hay duplicidades en el dataset
# Filtramos las filas donde la columna 'developer' contenga ese texto (sin importar mayúsculas/minúsculas)
resultados = df_bueno[df_bueno['Developers'].str.contains("Capcom", case = False, na = False)]
print("Nombres exactos encontrados en el dataset:")
print(resultados['Developers'].unique())

Nombres exactos encontrados en el dataset:
<StringArray>
[                  'Capcom Vancouver',                             'Capcom',
                   'CAPCOM Co., Ltd.',             'Capcom,Digital Eclipse',
                    'CAPCOM CO., LTD',                'Capcom U.S.A., Inc.',
 'Capcom Game Studio Vancouver, Inc.',                             'CAPCOM',
                   'CAPCOM CO., LTD.',       'Capcom Game Studio Vancouver',
          'Double Helix Games,Capcom',           'CAPCOM Co., Ltd.,GOG.com']
Length: 12, dtype: str


In [11]:
def limpiar_nombre_developer(nombre): # Si la celda está vacía, es nula o no es texto, devolvemos NaN
    if not isinstance(nombre, str) or nombre.strip() == "":
        return np.nan
    nombre = nombre.lower().strip() # Pasamos todo a minúsculas y quitamos espacios extraños
    partes = nombre.split(',') # Si hay múltiples developers separados por coma, los limpiamos por separado para no mezclar developers
    partes_limpias = []
    for parte in partes:
        p = parte.strip()
        p = ''.join(c for c in unicodedata.normalize('NFD', p) if unicodedata.category(c) != 'Mn') # Eliminar acentos y diacríticos
        p = p.replace('®', '').replace('™', '').replace('©', '') # Eliminar marcas registradas y símbolos comunes
        p = p.replace('.', '') # Quitar puntos (ej: "co. ltd." -> "co ltd", "s.r.l." -> "srl")
        p = re.sub(r'[-–—\/_]', ' ', p) # Reemplazar guiones y barras por espacios
        p = re.sub(r'[^a-z0-9\s]', '', p) # Eliminar el resto de caracteres especiales no deseados
        patron_limpieza = ( # Quitar siglas y palabras comunes
            r'\b(co|ltd|inc|gmbh|llc|sa|corp|corporation|holdings|sl|srl|pvt|association|group|'
            r'games|game|studios|studio|entertainment|interactive|software|digital|media|the)\b'
        )
        p = re.sub(patron_limpieza, '', p)
        p = ' '.join(p.split()) # Quitar espacios dobles que queden tras la limpieza
        if p: # Si tras la limpieza queda algo de texto
            partes_limpias.append(p.title()) # Formato "Título" (Capcom, Ubisoft)
    if not partes_limpias: # Si después de limpiar el texto se queda vacío (ej: la celda era solo "Co., Ltd.")
        return np.nan
    return ','.join(partes_limpias) # Unimos con una coma limpia sin espacios (estándar del Excel)

df_bueno['developer_clean'] = df_bueno['Developers'].apply(limpiar_nombre_developer)
df_bueno['publisher_clean'] = df_bueno['Publishers'].apply(limpiar_nombre_developer)
print(df_bueno[df_bueno['Developers'].str.contains('KOEI TECMO', case = False, na = False)]['developer_clean'].unique())
print(df_bueno[df_bueno['Developers'].str.contains('Capcom', case = False, na = False)]['developer_clean'].unique())

<StringArray>
[                         'Koei Tecmo',              'Square Enix,Koei Tecmo',
               'Koei Tecmo,Line,Motif', 'Platinumgames Team Ninja Koei Tecmo',
               'Team Ninja,Koei Tecmo']
Length: 5, dtype: str
<StringArray>
[   'Capcom Vancouver',              'Capcom',      'Capcom,Eclipse',
          'Capcom Usa', 'Double Helix,Capcom',       'Capcom,Gogcom']
Length: 6, dtype: str


In [13]:
# Buscamos posibles duplicados usando una comparación difusa (fuzzy matching). Agrupamos nombres que se parezcan un 82% o más
def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

counts = df_bueno['developer_clean'].value_counts()
todos_los_devs = [dev for dev in counts.index.tolist() if isinstance(dev, str)]
duplicados_sospechosos = set()
indice_palabras = defaultdict(list)
for dev in todos_los_devs:
    palabras = set(re.split(r'[\s,]+', dev.lower()))
    for palabra in palabras:
        if len(palabra) > 2: # Evitamos letras sueltas o palabras de 2 letras
            indice_palabras[palabra].append(dev)

for palabra, num_devs in indice_palabras.items():
    if len(num_devs) < 2: # Si solo hay un desarrollador con esa palabra, no hay duplicado
        continue
    if len(num_devs) > 40: # Evitamos palabras ultra-comunes que queden en el dataset
        continue
    for i in range(len(num_devs)):
        for j in range(i + 1, len(num_devs)):
            dev1 = num_devs[i]
            dev2 = num_devs[j]
            pareja_id = tuple(sorted([dev1, dev2]))
            if pareja_id in duplicados_sospechosos:
                continue
            nums1 = set(re.findall(r'\d+', dev1)) # Si tienen números diferentes (ej: 'Version 5' vs 'Version 6'), no se unifican
            nums2 = set(re.findall(r'\d+', dev2))
            if nums1 and nums2 and nums1 != nums2:
                continue  # Salta a la siguiente pareja directamente
            ratio = similar(dev1, dev2)
            if ratio >= 0.82: 
                duplicados_sospechosos.add(pareja_id)

resultado_final = [(d1, d2, similar(d1, d2)) for d1, d2 in duplicados_sospechosos]
resultado_final.sort(key = lambda x: x[2], reverse = True)
for dev1, dev2, score in resultado_final:
    print(f"¿Unificar? -> '{dev1}' con '{dev2}' (Similitud: {score:.2f})")

¿Unificar? -> 'Trend Readaktions Und Verlagsgesellschaft Mbh' con 'Trend Redaktions Und Verlagsgesellschaft Mbh' (Similitud: 0.99)
¿Unificar? -> 'Italian Party Of Indie Developer' con 'Italian Party Of Indie Developers' (Similitud: 0.98)
¿Unificar? -> 'Metthink,Matheus Reis Queiroga' con 'Metthink,Mattheus Reis Queiroga' (Similitud: 0.98)
¿Unificar? -> 'Kanuni Oyun Yazilim Ve Pazarlama Ticaret Limited Sirketi' con 'Kanuni Oyun Yazlm Ve Pazarlama Ticaret Limited Sirketi' (Similitud: 0.98)
¿Unificar? -> 'Just Add Water Development' con 'Just Add Water Developments' (Similitud: 0.98)
¿Unificar? -> 'Hong Haryong,Jung Youngju' con 'Hong Haryong,Jung Younju' (Similitud: 0.98)
¿Unificar? -> 'Laush Dmitriy Segeevich' con 'Laush Dmitriy Sergeevich' (Similitud: 0.98)
¿Unificar? -> 'Infogames Europe' con 'Infogrames Europe' (Similitud: 0.97)
¿Unificar? -> 'Age Zero,Purfect' con 'Age Zero,Purrfect' (Similitud: 0.97)
¿Unificar? -> 'Icebyte,Contendo' con 'Icebytes,Contendo' (Similitud: 0.97)
¿Unific

In [14]:
# Obtenemos los 100 desarrolladores más frecuentes como "semillas" principales
top_devs = df_bueno['developer_clean'].value_counts().head(100).index.tolist()
top_devs = [d for d in top_devs if isinstance(d, str) and len(d) > 3]
print("🔍 Buscando ramificaciones de grandes estudios (Casos tipo Capcom):\n")
for dev_principal in top_devs: # Expresión regular para buscar el nombre del dev aislado como palabra
    patron = rf'\b{re.escape(dev_principal)}\b' # Buscamos coincidencias en todo el dataset
    coincidencias = df_bueno[df_bueno['developer_clean'].str.contains(patron, case = False, na = False)]['developer_clean'].unique()
    if len(coincidencias) > 1: # Si encontramos variantes (más allá del nombre exacto solo)
        print(f"🏢 Estudio Principal: '{dev_principal}'")
        print(f"   Variantes encontradas ({len(coincidencias)}): {list(coincidencias)}")
        print("-" * 50)

🔍 Buscando ramificaciones de grandes estudios (Casos tipo Capcom):

🏢 Estudio Principal: 'Laush Dmitriy Sergeevich'
   Variantes encontradas (3): ['Laush Dmitriy Sergeevich', 'Ilxam,Laush Dmitriy Sergeevich', 'Falco,Laush Dmitriy Sergeevich']
--------------------------------------------------
🏢 Estudio Principal: 'Creobit'
   Variantes encontradas (9): ['Creobit', 'Jerzie,Creobit', 'Workroom7,Creobit', 'Toyman,Creobit', 'Good,Creobit', 'Creobit,Workroom7', 'Ejaw,Creobit', 'Unibit,Creobit', 'Creobit,Gameon Production']
--------------------------------------------------
🏢 Estudio Principal: 'Gamesforgames'
   Variantes encontradas (6): ['Gamesforgames', 'Andreev Worlds,Gamesforgames', 'Space R,Gamesforgames', 'Maks Volegov,Gamesforgames', 'Succubella,Gamesforgames', 'Scale,Gamesforgames']
--------------------------------------------------
🏢 Estudio Principal: 'Koei Tecmo'
   Variantes encontradas (5): ['Koei Tecmo', 'Square Enix,Koei Tecmo', 'Koei Tecmo,Line,Motif', 'Platinumgames Team N

In [17]:
# Habiendo analizado todos los cambios, unificamos los developers. "Original", "Nuevo dev"
unificaciones_completas_lista = [
    # Grandes compañías
    ("Idea Factory,Compile Heart,Neilo", "Idea Factory,Compile Heart"),
    ("Idea Factory,Compile Heart,Sting", "Idea Factory,Compile Heart"),
    ("Idea Factory,Compile Heart,Tamsoft", "Idea Factory,Compile Heart"),
    ("Idea Factory,Compile Heart,Felistella", "Idea Factory,Compile Heart"),
    ("Compile Heart,Idea Factory,Ghostlight", "Idea Factory,Compile Heart"),
    ("Compile Heart,Idea Factory", "Idea Factory,Compile Heart"),
    ("E Funsoft", "Funsoft"),
    ("Qloc,Bandai Namco", "Bandai Namco"),
    ("Arc System Works,Aplus", "Arc System Works"),
    ("Arc System Works,Atlus", "Arc System Works"),
    ("Arc System Works,Dotemu", "Arc System Works"),
    ("Arc System Works,Toybox", "Arc System Works"),
    ("Fk,Arc System Works", "Arc System Works"),
    ("Nihon Falcom,Ph3", "Nihon Falcom"),
    ("Nihon Falcom,Engine Bv,Ph3", "Nihon Falcom"),
    ("Nihon Falcom,Ph3,Engine Bv", "Nihon Falcom"),
    ("Nippon Ichi,Ph3", "Nippon Ichi"),
    ("Grasshopper Manufacture,Playism", "Grasshopper Manufacture"),
    ("Kadokawa Grasshopper Manufacture", "Grasshopper Manufacture"),
    ("Grasshopper Manufacture,Engine Bv", "Grasshopper Manufacture"),
    ("Rockstar North,Rockstar Toronto", "Rockstar North"),

    # Consolidación Square Enix
    ("Square Enix Montreal", "Square Enix"),
    ("Square Enix,Dokidoki Groove Works", "Square Enix"),
    ("Square Enix,D4Enterprise", "Square Enix"),
    ("Square Enix,Acquire", "Square Enix"),
    ("Square Enix,Xeen", "Square Enix"),
    ("Square Enix,Tose", "Square Enix"),
    ("Square Enix,Gemdrops", "Square Enix"),
    ("Square Enix,Artdink", "Square Enix"),
    ("Square Enix,Popcap", "Square Enix"),
    ("Square Enix,Koei Tecmo", "Square Enix"),
    ("Square Enix,Hand", "Square Enix"),
    ("Square Enix,Tri Ace", "Square Enix"),
    ("Square Enix,M2", "Square Enix"),
    ("Square Enix,Applibot", "Square Enix"),
    ("Square Enix,Claytechworks", "Square Enix"),
    ("Square Enix,Soleil", "Square Enix"),
    ("Square Enix,Bullets", "Square Enix"),
    ("Square Enix,Gamestudio,Kai Graphics", "Square Enix"),
    ("Square Enix,Sonic Powered", "Square Enix"),
    ("Square Enix,Human Head", "Square Enix"),
    ("Square Enix,Racjin", "Square Enix"),
    ("Square Enix,Platinumgames", "Square Enix"),
    ("Square Enix,Historia", "Square Enix"),
    ("Square Enix,Dotemu Sas", "Square Enix"),
    ("Square Enix,Artepiazza", "Square Enix"),
    ("Square Enix,Toylogic", "Square Enix"),
    ("Square Enix,Lancarse", "Square Enix"),
    ("Square Enix,Arzest", "Square Enix"),
    ("Square Enix,Team Arcana", "Square Enix"),
    ("Square Enix,Hexadrive", "Square Enix"),
    ("Square Enix,Tactic", "Square Enix"),

    # Otros grandes estudios y franquicias
    ("Blender Foundation", "Blender"),
    ("Capcom Vancouver", "Capcom"),
    ("Capcom Usa", "Capcom"),
    ("Capcom,Eclipse", "Capcom"),
    ("Double Helix,Capcom", "Capcom"),
    ("Capcom,Gogcom", "Capcom"),
    ("Koei Tecmo,Line,Motif", "Koei Tecmo"),
    ("Platinumgames Team Ninja Koei Tecmo", "Koei Tecmo"),
    ("Team Ninja,Koei Tecmo", "Koei Tecmo"),
    ("Gaterooze,Ink,Microprose", "Microprose"),
    ("Microprose,Sensible", "Microprose"),
    ("Magnetic Scrolls,Microprose", "Microprose"),
    ("Microprose,Theory", "Microprose"),
    ("Deadly,Microprose", "Microprose"),
    ("Realtime,Microprose", "Microprose"),
    ("Why485,Microprose", "Microprose"),
    ("Mps Labs,Microprose", "Microprose"),
    ("Cyanide Montreal", "Cyanide"),
    ("Cyanide,Umix", "Cyanide"),
    ("Umix,Cyanide", "Cyanide"),
    ("Leikir,Cyanide", "Cyanide"),
    ("Cotton Candy Cyanide", "Cyanide"),
    ("Cyanide,Spiders", "Cyanide"),
    ("Black Shamrock,Cyanide", "Cyanide"),
    ("Cyanide Tea", "Cyanide"),
    ("Pyramid,G Mode", "G Mode"),
    ("G Mode,Atlus", "G Mode"),
    ("G Mode,Marvelous", "G Mode"),
    ("G Mode,Room6", "G Mode"),
    ("G Mode,Fromsoftware", "G Mode"),
    ("G Mode,Plus81,Nekoday", "G Mode"),
    ("Room6,G Mode", "G Mode"),
    ("G Mode,Hopemoon", "G Mode"),
    ("Sokpop Collective,Sokpop Collective", "Sokpop Collective"),
    ("Sokpop Collective,Tom Van Den Boogaart", "Sokpop Collective"),
    ("Adriaan De Jongh,Sokpop Collective", "Sokpop Collective"),
    ("Dnovel,Dnovel", "Dnovel"),
    ("Dnovel,Freeanimals", "Dnovel"),
    ("Freeanimals,Dnovel", "Dnovel"),
    ("Imposant,Dnovel", "Dnovel"),
    ("Studio2760,Dnovel", "Dnovel"),
    ("Hgmgame,Hgmgame", "Hgmgame"),
    ("Hede,Hede Engine", "Hede"),
    ("Hede,Fallalypse", "Hede"),
    ("Hede,Hexaluga Mines", "Hede"),
    ("Ik,Hede", "Hede"),
    ("Kg Studuo,Hede", "Hede"),

    # Casos de múltiples developers
    ("Atlas V,Serendipity Films,Wild Fang Films,Giioii,Newmatic,Maniamind", "Atlas V"),
    ("Atlasv,Serendipity Films,Wild Fang Films,Giioii,Vertical", "Atlas V"),
    ("Atlasv,Serendipity Films,Wild Fang Films,Giioii,Moonshine", "Atlas V"),
    ("Atlas V,Serendipity Films,Wild Fang Films,Giioii,Newmatic", "Atlas V"),
    ("Atlas V,Serendipity Films,Wild Fang Films,Giioii,Albyon", "Atlas V"),
    ("Tero Lunkka,Valkeala,Black Thug", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Ntstudios,Blackthug", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Valkeala,Blackthug,Ntstudios", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Valkeala,Ntstudios,Blackthug", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Valkeala,Ntstudios", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Valkeala,Blackthug", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Gangsta,Blackthug", "Tero Lunkka,Valkeala"),
    ("Valkeala,Tero Lunkka,Blackthug", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Blackthug", "Tero Lunkka,Valkeala"),
    ("Valkeala,Tero Lunkka,Kendall", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Valkeala,Kendall", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Valkeala,Raciebug", "Tero Lunkka,Valkeala"),
    ("Tero Lunkka,Valkeala,Blackthug,Ruskija Experience", "Tero Lunkka,Valkeala"),
    ("Valkeala,Tero Lunkka", "Tero Lunkka,Valkeala"),
    ("Out Of Park Developments,Minos", "Out Of Park Developments"),
    ("Game54,Out Of Park Developments", "Out Of Park Developments"),
    ("Out Of Park Developments,Game54", "Out Of Park Developments"),
    ("Travellers Tales,Tt", "Travellers Tales"),
    ("Tt,Travellers Tales", "Travellers Tales"),
    ("Tt Fusion,Travellers Tales", "Travellers Tales"),
    ("Travellers Tales,Tt,Feral Mac", "Travellers Tales"),
    ("Travellers Tales,Feral Mac", "Travellers Tales"),
    ("Q,Double Eleven", "Double Eleven"),
    ("Double Eleven,Q", "Double Eleven"),
    ("Tt,Double Eleven", "Double Eleven"),
    ("Creative Assembly,Feral Mac", "Creative Assembly"),
    ("Creative Assembly,Feral", "Creative Assembly"),
    ("Relic,Feral Mac,Feral Linux", "Relic"),
    ("Eidos Montreal,Feral Mac", "Eidos Montreal"),
    ("Eidos Montreal,Feral Linux Mac", "Eidos Montreal"),
    ("Crystal Dynamics,Eidos Montreal,Feral Mac,Feral Linux,Nixxes", "Crystal Dynamics"),
    ("Crystal Dynamics,Eidos Montreal,Feral Mac,Nixxes", "Crystal Dynamics"),
    ("Aspyr,Crystal Dynamics", "Crystal Dynamics"),
    ("Qloc,Fromsoftware", "Fromsoftware"),
    ("Tequila Works,Qloc", "Tequila Works"),
    ("Naughty Dog,Nixxes,Iron Galaxy", "Naughty Dog"),
    ("Naughty Dog,Iron Galaxy", "Naughty Dog"),
    ("Infinity Ward,Raven,Beenox,Treyarch,High Moon,Sledgehammer,Activision Shanghai,Demonware,Toys For Bob", "Infinity Ward"),
    ("Infinity Ward,Raven,Beenox,Treyarch,High Moon,Sledgehammer,Demonware,Toys For Bob", "Infinity Ward"),
    ("Skybox Labs,Hidden Path,Ensemble,Forgotten Empires", "Skybox Labs"),
    ("Skybox Labs,Ensemble,Forgotten Empires", "Skybox Labs"),
    ("Gearbox,Aspyr Mac,Aspyr Linux", "Gearbox"),
    ("Firaxis,Aspyr Mac,Aspyr Linux", "Firaxis"),
    ("Graphite Lab,Joystick,Sonamu", "Graphite Lab"),
    ("Graphite Lab,Joystick", "Graphite Lab"),
    ("Stormregion,Thq Nordic", "Thq Nordic"),
    ("Innoglow,Stormregion,Thq Nordic", "Thq Nordic"),
    ("Patrones Escondites,Tlr", "Patrones Escondites"),
    ("Mizoriot Creative Company,Fun2", "Mizoriot Creative Company"),
    ("Wanin International,Jera", "Wanin International"),
    ("Dreams Uncorporated,Syck", "Dreams Uncorporated"),
    ("David S Womack Productions,Womack", "David S Womack Productions"),
    ("Cateia,Autumn Moon,Jordan Freeman", "Autumn Moon"),
    ("Autumn Moon,Jordan Freeman", "Autumn Moon"),
    ("Revolab,Ivan Zanottis Mymadnessworks", "Ivan Zanottis Mymadnessworks"),
    ("Ivan Zanottis Mymadnessworks,Dinobird", "Ivan Zanottis Mymadnessworks"),
    ("Frontier Developments,Asobo", "Frontier Developments"),
    ("Frontier Developments,Aspyr Mac", "Frontier Developments"),
    ("Storia,Narrator,Simoncreative", "Narrator"),
    ("Sounding Stone,Meijin Zhu,Maoucat", "Sounding Stone"),
    ("Sounding Stone,Meijin Zhu", "Sounding Stone"),
    ("Andrew Augustine,Zach Augustine,Alex Bade", "Andrew Augustine"),
    ("Andrew Augustine,Zach Augustine", "Andrew Augustine"),
    ("Space Cat,Snowflame,Back To Basics Gaming", "Snowflame"),
    ("Snowflame,Back To Basics Gaming", "Snowflame"),
    ("Hexes Of War,C3I", "Hexes Of War"),
    ("Ladomu,Angels Blue,Digitalez", "Angels Blue"),
    ("Angels Blue,Digitalez", "Angels Blue"),
    ("Snk,Code Mystics", "Code Mystics"),
    ("Jozgamer,Rfl Art,Hatus Quemuel", "Jozgamer"),
    ("Jozgamer,Hatus Quemuel", "Jozgamer"),
    ("Riveted,Lock N Load Publishing", "Lock N Load Publishing"),
    ("Lock N Load Publishing,Hexdraw", "Lock N Load Publishing"),
    ("Lockwood Publishing", "Lock N Load Publishing"),
    ("Brainwash Gang,Tlr", "Brainwash Gang"),
    ("Fayer,Sketchy Ceviche", "Sketchy Ceviche"),
    ("Spielmannspiel,Bison,Jujibla", "Spielmannspiel"),
    ("Spielmannspiel,Bison", "Spielmannspiel"),
    ("Humza Shakoor,Vejay Fongchoy,Thegoodmen Gamingstudio", "Thegoodmen Gamingstudio"),
    ("Humza Shakoor,Thegoodmen Gamingstudio", "Thegoodmen Gamingstudio"),
    ("Chibig,Talpa,Undercoders", "Undercoders"),
    ("Talpa,Undercoders", "Undercoders"),
    ("Nesh,Siluman Soft", "Siluman Soft"),
    ("Lucas Le Slo,Taletronic,Sergi Sabater", "Lucas Le Slo"),
    ("Lucas Le Slo,Sergi Sabater", "Lucas Le Slo"),
    ("Nama,Gotcha Gotcha", "Gotcha Gotcha"),
    ("Zink,Novabeargames", "Novabeargames"),
    ("Gentledriver,Genkung,Siki2046", "Gentledriver"),
    ("Gentledriver,Siki2046", "Gentledriver"),
    ("Gtai,Gentledriver", "Gentledriver"),
    ("Gentledriver,Era", "Gentledriver"),
    ("Team Gr,Incandescent Workshop", "Incandescent Workshop"),
    ("Mydreamforever Old,Nehaew", "Mydreamforever Old"),
    ("United Independent,Crafty", "United Independent"),
    ("Signal Space Lab,Actrio", "Signal Space Lab"),
    ("Gagonfe,Hugobdesigner,Ulon", "Gagonfe"),
    ("Gagonfe,Hugobdesigner", "Gagonfe"),
    ("Gagonfe,Delta,Rhowsl", "Gagonfe"),
    ("Gagonfe,Ulon,Rhowsl", "Gagonfe"),
    ("Netherrealm,Qloc,Shiver", "Netherrealm"),
    ("Netherrealm,Qloc", "Netherrealm"),
    ("Old School Vibes,Motion", "Old School Vibes"),
    ("Octato,Massive Miniteam", "Massive Miniteam"),
    ("Urbanm,Jellyfish Parade", "Jellyfish Parade"),
    ("Hyperboreangames,Svarog", "Hyperboreangames"),
    ("Independent Arts,Upjers", "Independent Arts"),
    ("Stefan Klaus,Stegalosaurus Development", "Stegalosaurus Development"),
    ("Lgt Sia,Https Wwwfacebookcom Lgtsia", "Lgt Sia"),
    ("Valve,Nexon", "Valve"),

    # Marcas, sufijos y errores
    ("Softstar Technologybeijingcoltd", "Softstar Technology Beijing"),
    ("Softstar Technologybeijing", "Softstar Technology Beijing"),
    ("Intergalaxy Entertaiment", "Inter Galaxy Entertaiment"),
    ("Redsquirrel Tribute", "Red Squirrel Tribute"),
    ("Immersive Vr Education Plc", "Immersive Vr Education"),
    ("Iv Productions,Trinity Team Srls", "Iv Productions,Trinity Team"),
    ("Zerouno,Operasoft", "Zerouno,Opera Soft"),
    ("Dept Of Silly Stuff", "Department Of Silly Stuff"),
    ("Bbc Applications Technologies Limited", "Bbc Applications Technologies"),
    ("Https Wwwsuperstargamescouk", "Superstar Games"),
    ("Wwwsuperstargamescouk", "Superstar Games"),
    ("Best Ride Simulators", "Best Simulators"),
    ("By Jan Zizka", "Jan Zizka"),
    ("By Cassandra", "Cassandra"),
    ("J Wgames", "Wgames"),
    ("A Guilda", "Guilda"),
    ("A Penguin", "Penguin"),
    ("0 Stamina", "Stamina"),
    ("1 Simple", "Simple"),
    ("Fm Simple", "Simple"),
    ("E Snail", "Snail"),
    ("E Sheep", "Sheep"),
    ("7 Raven", "Raven"),
    ("9 Squad", "Squad"),
    ("4 Hands", "Hands"),
    ("7 Funny", "Funny"),
    ("J Wonder", "Wonder"),
    ("Ws Net", "Ys Net"),
    ("D4 Laboratory", "Laboratory"),
    ("C Laboratory", "Laboratory"),
    ("Net Laboratory", "Laboratory"),
    ("Ssi,Strategic Simulations", "Strategic Simulations"),
    ("West Dragon Productions Dr", "West Dragon Productions"),
    ("Iracingcom Motorsport Simulations,Iracing", "Iracingcom Motorsport Simulations"),
    ("Source Byte Sp Z Oo,Nikodem Swider", "Source Byte"),
    ("Source Byte,Nikodem Swider", "Source Byte"),
    ("Jason Dale Swisher", "Jason Swisher"),
    ("Shawn Hirsch,Esq", "Shawn Hirsch"),
    ("Christopher Shove,Phd", "Christopher Shove"),
    ("Daniel Ritchie Pdhowlercom,Philip Staiger Thebest3Dcom", "Daniel Ritchie"),
    ("Daniel Ritchie Squirreldomecom,Philip Staiger Thebest3Dcom", "Daniel Ritchie"),
    ("Trend Readaktions Und Verlagsgesellschaft Mbh", "Trend Redaktions Und Verlagsgesellschaft Mbh"),
    ("Italian Party Of Indie Developer", "Italian Party Of Indie Developers"),
    ("Just Add Water Development", "Just Add Water Developments"),
    ("Ripknot System", "Ripknot Systems"),
    ("Ancient Pixel", "Ancient Pixels"),
    ("Phantom Pixel", "Phantom Pixels"),
    ("Growing Seed", "Growing Seeds"),
    ("Broken Dream", "Broken Dreams"),
    ("Pillow Fight", "Pillow Fights"),
    ("Gentle Giant", "Gentle Giants"),
    ("Silent Dream", "Silent Dreams"),
    ("Barking Dog", "Barking Dogs"),
    ("Binary Soul", "Binary Souls"),
    ("Lucid Dream", "Lucid Dreams"),
    ("Silver Cat", "Silver Cats"),
    ("Killer Bee", "Killer Bees"),
    ("Devil Dev", "Devil Devs"),
    ("That Guy", "That Guys"),
    ("Grease Monkey", "Grease Monkeys"),
    ("Electric Monk", "Electric Monkeys"),
    ("Furry Tails", "Furry Tale"),
    ("Metthink,Matheus Reis Queiroga", "Metthink"),
    ("Metthink,Mattheus Reis Queiroga", "Metthink"),
    ("Hong Haryong,Jung Youngju", "Hong Haryong"),
    ("Hong Haryong,Jung Younju", "Hong Haryong"),
    ("Laush Dmitriy Segeevich", "Laush Dmitriy Sergeevich"),
    ("Ip Izmailov Vladimir Yurievich", "Ip Izmailov Vladimir Yurievich"),
    ("Ip Izmaliov Vladimir Yurievich", "Ip Izmailov Vladimir Yurievich"),
    ("Ahmet Yigitbas", "Ahmet Yigitbasi"),
    ("Zhan Xiaojiao", "Zhang Xiaojiao"),
    ("Veysel Cnar", "Veysel Cinar"),
    ("Knut Muller", "Knut Mueller"),
    ("Lv Yannan,Sun Xin,Zhang Jianhui", "Lyu Yannan,Sun Xin,Zhang Jianhui"),
    ("Seebacher Noebert", "Seebacher Norbert"),
    ("Kipsaimmike Delgado", "Kipsaim Mike Delgado"),
    ("Syahir Bahiran Bin Hilmi", "Syahir Bahiran Hilmi"),
    ("Liao Wei Bo", "Liao Weibo"),
    ("Yoozoo Singapore Pte", "Youzu Singapore Pte"),
    ("Modesto A Rabena", "Modesto Rabena"),
    ("James B Stringer", "James Stringer"),
    ("Jonathan B Wood", "Jonathan Wood"),
    ("David J Ozsvath", "David Ozsvath"),
    ("Carlos Canizares Garcia", "Carlos Canizares"),
    ("Visual Arts Key", "Visualarts Key"),
    ("Quantic Dream Sas", "Quantic Dream")
]

mapeo_unificaciones = dict(unificaciones_completas_lista) # Convertimos la lista de tuplas en un diccionario de mapeo
df_bueno['developer_clean'] = df_bueno['developer_clean'].replace(mapeo_unificaciones)
df_bueno['publisher_clean'] = df_bueno['publisher_clean'].replace(mapeo_unificaciones)
total_devs_unicos = df_bueno['developer_clean'].nunique() # Número final de developers unificados
print(f"Número de developers original: {df['Developers'].nunique()}")
print(f"Tras la limpieza masiva, quedan {total_devs_unicos} desarrolladores distintos en el dataset.")

Número de developers original: 72859
Tras la limpieza masiva, quedan 66642 desarrolladores distintos en el dataset.


In [21]:
# Normalización de las nuevas variables developer_clean y publisher_clean para la lectura del modelo, y limpieza de valores nulos. Suavizado Bayesiano
media_global_exito = df_bueno['exito_target'].mean()
m = 5 # Peso, 5 juegos para alejarse de la media global

# Fórmula Bayesiana para developers
stats_devs = df_bueno.groupby('developer_clean').agg(exito_dev = ('exito_target', 'mean'), n_juegos = ('exito_target', 'count'))
stats_devs['dev_exito_bayesiano'] = ((stats_devs['n_juegos'] * stats_devs['exito_dev']) + (m * media_global_exito))/(stats_devs['n_juegos'] + m)
df_bueno['dev_exito_promedio'] = df_bueno['developer_clean'].map(stats_devs['dev_exito_bayesiano'])
df_bueno['dev_volumen_juegos'] = df_bueno['developer_clean'].map(stats_devs['n_juegos'])

# Suavizado Bayesiano para publishers
stats_pubs = df_bueno.groupby('publisher_clean').agg(exito_pub = ('exito_target', 'mean'), n_juegos = ('exito_target', 'count'))
stats_pubs['pub_exito_bayesiano'] = ((stats_pubs['n_juegos'] * stats_pubs['exito_pub']) + (m * media_global_exito))/(stats_pubs['n_juegos'] + m)
df_bueno['pub_exito_promedio'] = df_bueno['publisher_clean'].map(stats_pubs['pub_exito_bayesiano'])
df_bueno['pub_volumen_juegos'] = df_bueno['publisher_clean'].map(stats_pubs['n_juegos'])

# Limpieza de valores nulos
df_bueno['dev_exito_promedio'] = df_bueno['dev_exito_promedio'].fillna(media_global_exito)
df_bueno['dev_volumen_juegos'] = df_bueno['dev_volumen_juegos'].fillna(0)
df_bueno['pub_exito_promedio'] = df_bueno['pub_exito_promedio'].fillna(media_global_exito)
df_bueno['pub_volumen_juegos'] = df_bueno['pub_volumen_juegos'].fillna(0)

# Lista de columnas a eliminar definitivamente
columnas_a_borrar = [
    'Release date', # Ya extraímos año y mes
    'Estimated owners', # Ya creamos 'exito_target' a partir de ella
    'owners_promedio', # Columna numérica intermedia, no la queremos para entrenar
    'Supported languages', 'Genres', # Ya tenemos 'num_idiomas' y 'num_generos'
    'Publishers', 'Developers', # Ya tenemos 'tipo_estudio' y el conteo histórico
    'developer_clean', 'publisher_clean' # Hizimos la limpieza masiva y lo pasamos a las variables éxito promedio y volumen juegos
]

# Aplicamos el drop sobre el DataFrame y comprobamos las columnas definitivas con las que se entrenará el modelo
df_modelo = df_bueno.drop(columns = columnas_a_borrar)
print("Columnas finales para el modelo:")
print(df_modelo.columns.tolist())

Columnas finales para el modelo:
['Required age', 'Price', 'Windows', 'Mac', 'Linux', 'exito_target', 'num_idiomas', 'num_generos', 'año_lanzamiento', 'mes_lanzamiento', 'lanzamiento_navidad', 'juegos_publicados_historico', 'tipo_estudio', 'dev_exito_promedio', 'dev_volumen_juegos', 'pub_exito_promedio', 'pub_volumen_juegos']


In [25]:
cols_os = ['Windows', 'Mac', 'Linux']
df_modelo[cols_os] = df_modelo[cols_os].astype(int) # Convertimos directamente True/False a 1/0
df_modelo.head()

,Required age,Price,Windows,Mac,Linux,exito_target,num_idiomas,num_generos,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,juegos_publicados_historico,tipo_estudio,dev_exito_promedio,dev_volumen_juegos,pub_exito_promedio,pub_volumen_juegos
Name,,,,,,,,,,,,,,,,,
Black Dragon Mage Playtest,0,0.00,1,0,0,0,1,1,2023,8,0,1,0,0.245860,0.0,0.245860,0.0
Supipara - Chapter 1 Spring Has Come!,0,5.24,1,0,0,0,1,1,2016,7,0,77,2,0.171485,8.0,0.210113,77.0
Mystery Solitaire The Black Raven,0,4.99,1,1,0,0,4,1,2019,5,0,267,2,0.056391,70.0,0.097798,345.0
버튜버 파라노이아 - Vtuber Paranoia,0,8.99,1,0,0,0,1,3,2024,10,0,13,1,0.245860,0.0,0.245860,0.0
Maze Quest VR,0,4.99,1,0,0,0,1,2,2025,4,0,1,0,0.175614,2.0,0.204884,1.0


Reducción de Dimensionalidad y Eliminación de Redundancias:
Una vez finalizada la ingeniería de variables, se procedió a depurar el DataFrame eliminando las columnas cualitativas originales de texto plano (Genres, Publishers, Supported languages) y los campos intermedios de negocio (Release date, Estimated owners). Este paso es crítico para consolidar una matriz de características puramente numérica, optimizar el uso de memoria en el entorno de desarrollo y mitigar el riesgo de data leakage durante el entrenamiento del modelo.

In [27]:
df_modelo.info()

<class 'pandas.DataFrame'>
Index: 124644 entries, Black Dragon Mage Playtest to Project Liminal 2
Data columns (total 17 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Required age                 124644 non-null  int64  
 1   Price                        124644 non-null  float64
 2   Windows                      124644 non-null  int64  
 3   Mac                          124644 non-null  int64  
 4   Linux                        124644 non-null  int64  
 5   exito_target                 124644 non-null  int64  
 6   num_idiomas                  124644 non-null  int64  
 7   num_generos                  124644 non-null  int64  
 8   año_lanzamiento              124644 non-null  int32  
 9   mes_lanzamiento              124644 non-null  int64  
 10  lanzamiento_navidad          124644 non-null  int64  
 11  juegos_publicados_historico  124644 non-null  int64  
 12  tipo_estudio                 124644 no

In [28]:
# Separamos las características (X) y el objetivo (y) antes de normalizar
X = df_modelo.drop(columns = ['exito_target'])
y = df_modelo['exito_target']

# Estandarizamos las variables numéricas continuas (Media = 0, Varianza = 1). Solo escalamos las que no son booleanas/binarias
columnas_a_escalar = [
    'Price', 'Required age', 'num_idiomas', 'num_generos', 'juegos_publicados_historico', 'año_lanzamiento',
    'dev_exito_promedio', 'dev_volumen_juegos', 'pub_exito_promedio', 'pub_volumen_juegos'
]
scaler = StandardScaler()
X[columnas_a_escalar] = scaler.fit_transform(X[columnas_a_escalar])
print(f"Dataset listo. Características (X): {X.shape}, Target (y): {y.shape}")
X.head()

Dataset listo. Características (X): (124644, 16), Target (y): (124644,)


,Required age,Price,Windows,Mac,Linux,num_idiomas,num_generos,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,juegos_publicados_historico,tipo_estudio,dev_exito_promedio,dev_volumen_juegos,pub_exito_promedio,pub_volumen_juegos
Name,,,,,,,,,,,,,,,,
Black Dragon Mage Playtest,-0.098826,-0.384657,1,0,0,-0.318697,-1.267932,0.407933,8,0,-0.314067,0,-0.088693,-0.356671,-0.146746,-0.335845
Supipara - Chapter 1 Spring Has Come!,-0.098826,0.033542,1,0,0,-0.318697,-1.267932,-1.714291,7,0,1.202642,2,-0.511396,0.012962,-0.309889,1.144637
Mystery Solitaire The Black Raven,-0.098826,0.013590,1,1,0,-0.096888,-1.267932,-0.804767,5,0,4.994414,2,-1.165516,2.877619,-0.822479,6.297485
버튜버 파라노이아 - Vtuber Paranoia,-0.098826,0.332827,1,0,0,-0.318697,0.175330,0.711108,10,0,-0.074587,1,-0.088693,-0.356671,-0.146746,-0.335845
Maze Quest VR,-0.098826,0.013590,1,0,0,-0.318697,-0.546301,1.014282,4,0,-0.314067,0,-0.487925,-0.264263,-0.333758,-0.316618


Tratamiento de Valores Críticos y Estandarización de Escalas:

Como último filtro de robustez, se interceptaron posibles valores infinitos (inf / -inf) transformándolos a nulos y aplicando una política de tolerancia cero mediante la imputación final a 0.
Para mitigar el impacto de las diferencias de magnitud entre las variables (por ejemplo, el rango de precios frente a las variables booleanas de plataformas), se aplicó un escalado empleando un StandardScaler. Este procedimiento transforma las distribuciones de las variables continuas para que presenten una media igual a $0$ y una varianza unitaria ($1$), garantizando la estabilidad numérica y un cálculo óptimo de las distancias en los algoritmos basados en gradientes y árboles.

Validación de la Política de Imputación frente a la Media Aritmética:

Se evaluó la viabilidad de utilizar la media estadística para la sustitución de valores ausentes, descartándose en favor de una imputación lógica por variables. Imputar la media en características como Price o Required age alteraría artificialmente la naturaleza de los datos (transformando juegos Free-to-Play en títulos de precio promedio, o introduciendo restricciones de edad inexistentes). Por lo tanto, se optó por una estrategia de valores mínimos absolutos (0 y 1), preservando la semántica del negocio de los videojuegos y evitando el sesgo en la distribución de las variables predictoras.

Interpretación del Espacio de Características Estandarizado:

La presencia de valores negativos en la matriz de características estandarizada ($X$) no representa una anomalía en los datos de origen (como precios o edades negativas), sino que es el resultado intrínseco de la transformación mediante StandardScaler. Al desplazar el centro de la distribución a una media aritmética de $\mu = 0$, el signo del valor estandarizado actúa como un indicador posicional: los valores negativos representan registros que se sitúan cuantitativamente por debajo de la media global del mercado de Steam, mientras que los valores positivos denotan registros que superan dicho promedio.

In [30]:
X.describe()

,Required age,Price,Windows,Mac,Linux,num_idiomas,num_generos,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,juegos_publicados_historico,tipo_estudio,dev_exito_promedio,dev_volumen_juegos,pub_exito_promedio,pub_volumen_juegos
count,1.246440e+05,1.246440e+05,124644.000000,124644.000000,124644.000000,1.246440e+05,1.246440e+05,1.246440e+05,124644.000000,124644.000000,1.246440e+05,124644.000000,1.246440e+05,1.246440e+05,1.246440e+05,1.246440e+05
mean,5.974205e-17,-2.918695e-17,0.999647,0.172475,0.127515,-5.974205e-17,7.547564e-17,5.910358e-15,6.671954,0.176206,3.921997e-17,0.308077,1.240446e-16,-4.104416e-18,9.941807e-17,3.283532e-17
std,1.000004e+00,1.000004e+00,0.018785,0.377794,0.333551,1.000004e+00,1.000004e+00,1.000004e+00,3.462484,0.380997,1.000004e+00,0.628285,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00
min,-9.882576e-02,-3.846571e-01,0.000000,0.000000,0.000000,-3.186967e-01,-1.267932e+00,-7.474613e+00,1.000000,0.000000,-3.140674e-01,0.000000,-1.456526e+00,-3.566708e-01,-1.245142e+00,-3.358450e-01
25%,-9.882576e-02,-3.375697e-01,1.000000,0.000000,0.000000,-3.186967e-01,-5.463008e-01,-5.015919e-01,4.000000,0.000000,-3.140674e-01,0.000000,-4.879249e-01,-3.104667e-01,-4.673371e-01,-3.166179e-01
50%,-9.882576e-02,-1.939134e-01,1.000000,0.000000,0.000000,-3.186967e-01,1.753304e-01,4.079327e-01,7.000000,0.000000,-2.941107e-01,0.000000,-3.215782e-01,-2.642626e-01,-3.337576e-01,-2.973909e-01
75%,-9.882576e-02,6.147565e-02,1.000000,0.000000,0.000000,-9.688767e-02,8.969615e-01,7.111076e-01,10.000000,0.000000,-1.943272e-01,0.000000,9.773512e-02,-1.718543e-01,8.497451e-02,-2.012557e-01
max,1.292957e+01,7.942282e+01,1.000000,1.000000,1.000000,7.222811e+00,1.172143e+01,1.317457e+00,12.000000,1.000000,1.072199e+01,2.000000,7.372454e+00,1.036269e+01,6.130382e+00,1.031594e+01


In [16]:
# 6. Entrenamiento del modelo y prueba de algoritmos

In [8]:
# 7. Creación del dataset de predicciones futuras. https://steamdb.info/app/4824610/